In [1]:
pip install hmmlearn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Roaa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ==================== HMM PREPROCESSING & MODELING ====================
import numpy as np
import pandas as pd
from hmmlearn import hmm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import gc

In [3]:
# ==================== 1. LOAD PREPROCESSED DATA ====================
SAVE_PATH = "windowed_data"
NUM_BATCHES = 5

X_all = []
y_all = []

for b in range(NUM_BATCHES):
    X_batch = np.load(f"{SAVE_PATH}/X_batch_{b}.npy")
    y_batch = np.load(f"{SAVE_PATH}/y_batch_{b}.npy")
    X_all.append(X_batch)
    y_all.append(y_batch)

X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0).flatten()

print(f"Total samples: {len(X_all)}")
print(f"Shape: X={X_all.shape}, y={y_all.shape}")

Total samples: 164424
Shape: X=(164424, 24), y=(164424,)


In [4]:
# ==================== 2. SPLIT DATA ====================
n_samples = len(X_all)
train_size = int(0.70 * n_samples)
val_size = int(0.15 * n_samples)

X_train = X_all[:train_size]
y_train = y_all[:train_size]

X_val = X_all[train_size:train_size + val_size]
y_val = y_all[train_size:train_size + val_size]

X_test = X_all[train_size + val_size:]
y_test = y_all[train_size + val_size:]

print(f"Train: {len(X_train)} samples")
print(f"Val: {len(X_val)} samples")
print(f"Test: {len(X_test)} samples")

del X_all, y_all
gc.collect()

Train: 115096 samples
Val: 24663 samples
Test: 24665 samples


110

In [5]:
# ==================== 3. DISCRETIZE GLUCOSE VALUES ====================
# Define glucose ranges (clinically relevant)
def discretize_glucose(values):
    """
    Convert continuous glucose to discrete states:
    0: Very Low (40-70)
    1: Low (70-100)
    2: Normal (100-140)
    3: High (140-180)
    4: Very High (180-400)
    """
    states = np.zeros(values.shape, dtype=int)
    states[(values >= 40) & (values < 70)] = 0
    states[(values >= 70) & (values < 100)] = 1
    states[(values >= 100) & (values < 140)] = 2
    states[(values >= 140) & (values < 180)] = 3
    states[(values >= 180) & (values <= 400)] = 4
    return states

# Discretize sequences
X_train_discrete = np.apply_along_axis(discretize_glucose, 1, X_train)
X_val_discrete = np.apply_along_axis(discretize_glucose, 1, X_val)
X_test_discrete = np.apply_along_axis(discretize_glucose, 1, X_test)

y_train_discrete = discretize_glucose(y_train)
y_val_discrete = discretize_glucose(y_val)
y_test_discrete = discretize_glucose(y_test)

print(f"5 States: Very Low, Low, Normal, High, Very High")
print(f"Train states distribution: {np.bincount(y_train_discrete)}")

5 States: Very Low, Low, Normal, High, Very High
Train states distribution: [ 3861 13934 29373 26139 41789]


In [8]:
# ==================== 4. TRAIN HMM ====================
# Parameters
N_STATES = 5  # Hidden states
N_FEATURES = 5  # Observable states (glucose levels)

# Flatten sequences for training
train_sequences = X_train_discrete.reshape(-1, 1)
train_lengths = [X_train_discrete.shape[1]] * len(X_train_discrete)

print(f"Training HMM with {N_STATES} hidden states")

# Train Gaussian HMM
model_hmm = hmm.GaussianHMM(
    n_components=N_STATES,
    covariance_type="diag",
    n_iter=100,
    random_state=42,
    verbose=False
    
)

model_hmm.fit(train_sequences, lengths=train_lengths)

print(f"HMM trained successfully")
print(f"Converged: {model_hmm.monitor_.converged}")

del train_sequences, train_lengths
gc.collect()

Training HMM with 5 hidden states


Model is not converging.  Current: 17315754.39271499 is not greater than 17315754.602994453. Delta is -0.2102794647216797


HMM trained successfully
Converged: True


5623

In [13]:
# ==================== 5. PREDICTION FUNCTION ====================
def predict_next_state(model, sequence):
    """Predict next glucose state using HMM"""
    sequence = sequence.reshape(-1, 1)
    
    # Get hidden state probabilities
    hidden_states = model.predict(sequence)
    
    # Use last hidden state to predict next observation
    last_state = hidden_states[-1]
    
    # Get mean of the distribution for this state
    predicted_value = model.means_[last_state][0]
    
    return predicted_value

In [14]:
# ==================== 6. CONVERT STATES BACK TO GLUCOSE VALUES ====================
def state_to_glucose(state):
    """Convert discrete state back to glucose value (midpoint)"""
    ranges = {
        0: 55,   # Very Low: 40-70 → midpoint 55
        1: 85,   # Low: 70-100 → midpoint 85
        2: 120,  # Normal: 100-140 → midpoint 120
        3: 160,  # High: 140-180 → midpoint 160
        4: 290   # Very High: 180-400 → midpoint 290
    }
    return ranges.get(int(state), 120)

In [15]:
# ==================== 7. EVALUATE ON TEST SET ====================
y_pred_hmm = []

for i in range(len(X_test_discrete)):
    pred_state = predict_next_state(model_hmm, X_test_discrete[i])
    pred_state_discrete = int(np.round(pred_state))
    pred_state_discrete = np.clip(pred_state_discrete, 0, 4)
    pred_glucose = state_to_glucose(pred_state_discrete)
    y_pred_hmm.append(pred_glucose)

y_pred_hmm = np.array(y_pred_hmm)

# Calculate metrics
errors = y_test - y_pred_hmm
mse = np.mean(errors ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(errors))

# Accuracy within ±15 mg/dL
delta = 15
accuracy = (np.sum(np.abs(errors) <= delta) / len(errors)) * 100

# MAPE
mape = np.mean(np.abs(errors) / np.abs(y_test)) * 100
mape_accuracy = 100 - mape

print("HMM TEST RESULTS")
print(f"RMSE: {rmse:.2f} mg/dL")
print(f"MAE: {mae:.2f} mg/dL")
print(f"MSE: {mse:.2f}")
print(f"Accuracy (±15 mg/dL): {accuracy:.2f}%")
print(f"MAPE: {mape:.2f}%")
print(f"MAPE Accuracy: {mape_accuracy:.2f}%")

HMM TEST RESULTS
RMSE: 46.50 mg/dL
MAE: 32.41 mg/dL
MSE: 2162.51
Accuracy (±15 mg/dL): 44.56%
MAPE: 19.23%
MAPE Accuracy: 80.77%


In [16]:
# ==================== 8. BASELINE COMPARISON ====================
y_naive = X_test[:, -1]
errors_naive = y_test - y_naive
rmse_naive = np.sqrt(np.mean(errors_naive ** 2))
mae_naive = np.mean(np.abs(errors_naive))
acc_naive = (np.sum(np.abs(errors_naive) <= 15) / len(errors_naive)) * 100
mape_naive = np.mean(np.abs(errors_naive) / np.abs(y_test)) * 100

print(f"Naive RMSE: {rmse_naive:.2f} mg/dL")
print(f"Naive MAE: {mae_naive:.2f} mg/dL")
print(f"Naive Accuracy: {acc_naive:.2f}%")
print(f"Naive MAPE: {mape_naive:.2f}%")

Naive RMSE: 14.75 mg/dL
Naive MAE: 9.72 mg/dL
Naive Accuracy: 80.90%
Naive MAPE: 6.41%


In [17]:
# ==================== 9. VISUALIZATIONS ====================
# Comparison Table
comparison_df = pd.DataFrame({
    'Model': ['Naive Baseline', 'HMM'],
    'RMSE': [rmse_naive, rmse],
    'MAE': [mae_naive, mae],
    'Accuracy (%)': [acc_naive, accuracy],
    'MAPE (%)': [mape_naive, mape],
    'MAPE Accuracy (%)': [100 - mape_naive, mape_accuracy]
})

print("\n" + comparison_df.to_string(index=False))

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# RMSE
axes[0].bar(comparison_df['Model'], comparison_df['RMSE'], color=['gray', 'blue'])
axes[0].set_ylabel('RMSE (mg/dL)')
axes[0].set_title('RMSE Comparison')
axes[0].grid(axis='y', alpha=0.3)

# Accuracy
axes[1].bar(comparison_df['Model'], comparison_df['Accuracy (%)'], color=['gray', 'green'])
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy (±15 mg/dL)')
axes[1].grid(axis='y', alpha=0.3)

# MAPE Accuracy
axes[2].bar(comparison_df['Model'], comparison_df['MAPE Accuracy (%)'], color=['gray', 'orange'])
axes[2].set_ylabel('MAPE Accuracy (%)')
axes[2].set_title('MAPE Accuracy (100 - MAPE)')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('hmm_comparison.png', dpi=300, bbox_inches='tight')
print("Saved: hmm_comparison.png")
plt.close()

# Save results
comparison_df.to_csv('hmm_results.csv', index=False)
print("Saved: hmm_results.csv")

# Cleanup
del X_train, X_val, X_test, y_train, y_val, y_test
del X_train_discrete, X_val_discrete, X_test_discrete
gc.collect()


         Model      RMSE       MAE  Accuracy (%)  MAPE (%)  MAPE Accuracy (%)
Naive Baseline 14.749504  9.720819     80.904115  6.409017          93.590981
           HMM 46.502844 32.407703     44.561119 19.231068          80.768932
Saved: hmm_comparison.png
Saved: hmm_results.csv


81